Example 13-11. A function to read functions and classes from a notebook stored as a .py
file in a Git repository

In [ ]:
# notebook_test_utilities.py

import inspect
import os
import sys
from pathlib import Path

# if module is base for all your tests, you setting Spark should be here.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
# Define non-Python prefixes that should be ignored.
IGNORED_PREFIXES = ("%",)

# Read a Fabric notebook in Python format and extract
# its classes and functions for testing.
def read_fabric_notebook(
    notebook_folder: Path | str,
    injected_mocks: dict[str, object] | None = None,
) -> dict[str, object]:
    # Read the notebook and filter out lines starting
    # with ignored prefixes, such as %pip install.
    # Ensure notebook dependencies are installed
    # in the test environment.
    source = (Path(notebook_folder) / "notebook-content.py").read_text(encoding="utf-8")
    source = "\n".join(
        line
        for line in source.splitlines()
        if not line.strip().startswith(IGNORED_PREFIXES)
    )
    # Create a global dictionary with a test module name
    # and any injected mocks.
    exec_globals = {"__name__": "__notebook_test__", **(injected_mocks or {})}
    for module_name, module_mock in (injected_mocks or {}).items():
        sys.modules[module_name] = module_mock
    # Execute the notebook code so its contents can be
    # extracted and used in the test.
    exec(source, exec_globals)
    # Make functions and classes from the notebook available
    # to the calling test.
    caller_globals = inspect.currentframe().f_back.f_globals
    for name, value in exec_globals.items():
        if not name.startswith("_") and callable(value):
            caller_globals[name] = value

Example 13-12. A Python test script unit testing a function from a notebook item

In [ ]:
from pathlib import Path
import pytest
from pyspark.sql import SparkSession
from notebook_test_utilities import read_fabric_notebook

NOTEBOOK_FOLDER = Path(
    "solution/lakehouse_processing/process_from_bronze_to_silver.Notebook"
)


class NotebookUtilsMock:
    class credentials:
        @staticmethod
        def getSecret(*_args, **_kwargs):
            return "fake-secret"


@pytest.fixture(scope="module")
def spark() -> SparkSession:
    s = (
        SparkSession.builder.master("local[1]")
        .appName("test_bronze_to_silver_processing")
        .config("spark.ui.enabled", "false")
        .getOrCreate()
    )
    s.sparkContext.setLogLevel("ERROR")
    read_fabric_notebook(
        NOTEBOOK_FOLDER,
        injected_mocks={"notebookutils": NotebookUtilsMock()},
    )
    yield s
    s.stop()


def test_given_df_when_add_audit_columns_then_fields_and_values_are_set(spark):
    # Given: a base DataFrame and an audit value
    base_df = spark.createDataFrame([(1,), (2,), (3,)], ["n"])
    # When: we add the audit column
    df = add_audit_columns(base_df, "unit_test")
    # Then: the audit column is present
    assert "audit_timestamp" in df.columns
    assert "audit_source" in df.columns
    assert df.select("audit_source").distinct().collect()[0][0] == "unit_test"